<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Fish-S2-Pro_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fish Audio S2 Pro — 4B+400M Dual-AR TTS with 15K Inline Emotion Tags

Flagship open-weights TTS from [Fish Audio](https://fish.audio) (CCFA). 4 B-parameter slow transformer + 400 M fast transformer in a **Dual-Autoregressive** architecture over a 10-codebook RVQ codec at ~21 Hz. Trained on **10 M+ hours across 80+ languages**, RL-aligned with GRPO. Supports **15 000+ free-form inline emotion/prosody tags** (e.g. `[whisper]`, `[excited]`, `[low voice in small room]`, `[pitch up]`) plus native multi-speaker `<|speaker:N|>` and multi-turn conversation. `fish-audio-research-license` (free for research / non-commercial).

### Quick Start
1. Connect to a **GPU runtime** — needs ~24 GB VRAM (A100 40 GB ideal, L4 24 GB works, T4 16 GB will likely OOM)
2. Run **Steps 1–4** in order — no token, no sign-up needed
3. Use the Gradio UI for interactive work, or Steps 6–7 for one-off / batch generation
4. Cache persists in your Google Drive if you mount it (see Step 2)

### Misuse notice (from the model card)
Do not use this model to impersonate any individual without their explicit consent. The authors disclaim responsibility for unlawful or unethical downstream use. The model is **not** approved for production commercial use — see `fishaudio/fish-speech` `LICENSE` for terms.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones the [fish-speech](https://github.com/fishaudio/fish-speech) repo and installs runtime deps.
# @markdown Skipping torch downgrade (pyproject.toml pins torch==2.8.0 — incompatible with Colab torch 2.11).

import os, sys, subprocess, pathlib

# Clone repo
REPO_DIR = pathlib.Path("/content/fish-speech")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fishaudio/fish-speech.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# System deps
subprocess.run(["apt-get", "install", "-y", "portaudio19-dev", "libsox-dev", "ffmpeg", "-qq"],
               check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Install Fish-S2-Pro with --no-deps to prevent torch downgrade from 2.11 to 2.8
print("[Fish] installing fish-speech (--no-deps, keeping Colab torch 2.11) ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."],
    cwd=str(REPO_DIR), check=True)

# Install runtime deps from fish-speech pyproject.toml (no torch/torchaudio override)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers<=4.57.3", "lightning>=2.1.0", "hydra-core>=1.3.2",
     "einops>=0.7.0", "kui>=1.6.0", "pyrootutils>=1.0.4", "resampy>=0.4.3",
     "einx[torch]==0.2.2", "zstandard>=0.22.0", "pyaudio",
     "modelscope==1.17.1", "opencc-python-reimplemented==0.1.7",
     "silero-vad", "ormsgpack", "tiktoken>=0.8.0", "pydantic==2.9.2",
     "cachetools", "descript-audio-codec", "safetensors",
     "datasets==2.18.0", "tensorboard>=2.14.1",
     "loralib>=0.1.2", "wandb>=0.19.0", "grpcio>=1.58.0", "uvicorn>=0.30.0",
     "natsort>=8.4.0", "pydub", "loguru>=0.6.0",
     "numpy", "librosa>=0.10.1", "rich>=13.5.3",
     "gradio>5.0.0", "tqdm==4.66.5", "soundfile", "hf_transfer"],
    check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
    sys.path.insert(0, str(REPO_DIR / "tools"))
os.environ["EINX_FILTER_TRACEBACK"] = "false"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch, torchaudio, pyrootutils
pyrootutils.setup_root(str(REPO_DIR), indicator=".project-root", pythonpath=True)
if not torch.cuda.is_available():
    raise SystemExit("No GPU detected. Connect a GPU runtime: Runtime -> Change runtime type -> L4 or A100")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"[Fish] torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0)}  VRAM={vram_gb:.1f} GB")
if vram_gb < 20:
    print("[Fish] WARNING: <20 GB VRAM may OOM for the 4B model. T4 15GB will likely fail.")

os.makedirs("/content/audio_out", exist_ok=True)
os.makedirs("/content/ref_audio", exist_ok=True)
print("[Setup] /content/audio_out and /content/ref_audio ready.")


In [ ]:
# @title STEP 2 — Pre-cache Model Weights
# @markdown Downloads the 4B S2-Pro LLaMA weights + the DAC codec (~10 GB total) into
# @markdown `/content/drive/MyDrive/AEI_TTS_Cache/fish-speech/` if you mount Drive, otherwise `/content/fish-speech-cache/`.
# @markdown Subsequent runs reuse this directory. Skips re-downloading if already present.
import os, subprocess, pathlib, torch, shutil

MODEL_ID = "fishaudio/s2-pro"
LOCAL_DIR_NAME = "fish-speech"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = pathlib.Path("/content/drive/MyDrive/AEI_TTS_Cache")
    print("[Cache] using Google Drive at", BASE)
except Exception:
    BASE = pathlib.Path("/content") / f"{LOCAL_DIR_NAME}-cache"
    print("[Cache] Drive not available, using local", BASE)
BASE.mkdir(parents=True, exist_ok=True)
CKPT_DIR = BASE / LOCAL_DIR_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(BASE / 'huggingface'))
(BASE / "huggingface").mkdir(parents=True, exist_ok=True)

# Check both the legacy CKPT_DIR and the standard HF cache
need = []
hf_cache = pathlib.Path(os.environ.get("HF_HOME", "/root/.cache/huggingface")) / "hub"
for fname in ("config.json", "model.safetensors.index.json", "tokenizer.tiktoken", "special_tokens_map.json", "codec.pth"):
    if not (CKPT_DIR / fname).exists() and not (hf_cache / f"models--{MODEL_ID.replace('/', '--')}" / "snapshots").exists():
        need.append(fname)
if need:
    print(f"[Cache] missing {need} — downloading")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_ID,
        allow_patterns=["*.json", "*.tiktoken", "*.safetensors", "codec.pth", "*.txt", "*.md"],
        max_workers=4,
    )
else:
    print(f"[Cache] all files present in {CKPT_DIR}")
free_gb = shutil.disk_usage("/content").free / 1024**3
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1024**3) if torch.cuda.is_available() else 0
print(f"[Cache] disk free: {free_gb:.1f} GB   GPU VRAM: {vram_gb:.1f} GB")
if torch.cuda.is_available() and vram_gb < 18:
    print("[Cache] WARNING: S2-Pro typically needs ~20 GB of VRAM. The T4 (15 GB) will OOM.")

In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)

# @markdown Defines the inference wrappers. Model is loaded on first call to `get_engine()` and

# @markdown cached in a global. We use the official `TTSInferenceEngine` with the 4B LLaMA + DAC codec.

import os, sys, pathlib, tempfile, time

import numpy as np, torch, torchaudio, gradio as gr

import soundfile as sf

REPO_DIR = pathlib.Path("/content/fish-speech")

sys.path.insert(0, str(REPO_DIR))

sys.path.insert(0, str(REPO_DIR / "tools"))

import pyrootutils

pyrootutils.setup_root(str(REPO_DIR), indicator=".project-root", pythonpath=True)

from fish_speech.inference_engine import TTSInferenceEngine

from fish_speech.models.dac.inference import load_model as load_decoder_model

from fish_speech.models.text2semantic.inference import launch_thread_safe_queue

from fish_speech.utils.schema import ServeTTSRequest, ServeReferenceAudio

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PRECISION = torch.bfloat16

COMPILE = False

ENGINE = None

SAMPLE_RATE = None

def _ckpt_dir() -> pathlib.Path:

    env = os.environ.get("FISH_CKPT_DIR")

    if env:

        return pathlib.Path(env)

    # Try HF cache first (where snapshot_download in Step 2 actually puts files)
    try:
        from huggingface_hub import try_to_load_from_cache
        cached = try_to_load_from_cache("fishaudio/s2-pro", "config.json")
        if cached:
            return pathlib.Path(cached).parent
    except Exception:
        pass
    # Fall back to Drive or local cache
    try:
        from google.colab import drive
        if pathlib.Path("/content/drive").exists():
            return pathlib.Path("/content/drive/MyDrive/AEI_TTS_Cache/fish-speech")
    except Exception:
        pass
    return pathlib.Path("/content/fish-speech-cache/fish-speech")

CKPT_DIR = _ckpt_dir()

def get_engine():

    global ENGINE, SAMPLE_RATE

    if ENGINE is not None:

        return ENGINE

    if DEVICE == "cpu":

        raise RuntimeError("S2-Pro requires a CUDA GPU (T4 15GB will likely OOM, L4 24GB / A100 40GB recommended).")

    print(f"[Fish] loading LLaMA from {CKPT_DIR}")

    llama_queue = launch_thread_safe_queue(

        checkpoint_path=str(CKPT_DIR),

        device=DEVICE,

        precision=PRECISION,

        compile=COMPILE,

    )

    print("[Fish] loading DAC codec")

    decoder = load_decoder_model(

        config_name="modded_dac_vq",

        checkpoint_path=str(CKPT_DIR / "codec.pth"),

        device=DEVICE,

    )

    ENGINE = TTSInferenceEngine(

        llama_queue=llama_queue,

        decoder_model=decoder,

        precision=PRECISION,

        compile=COMPILE,

    )

    SAMPLE_RATE = decoder.sample_rate

    print(f"[Fish] engine ready — sample rate {SAMPLE_RATE} Hz")

    return ENGINE

def _materialize_ref_audio(audio):

    if audio is None:

        return None, None

    if isinstance(audio, str):

        return audio, None

    if isinstance(audio, tuple) and len(audio) == 2 and isinstance(audio[1], np.ndarray):

        sr, wav = audio

        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)

        sf.write(tmp.name, wav.astype(np.float32), int(sr))

        return tmp.name, None

    return None, None

def synthesize(text, ref_path, ref_text, max_new_tokens, chunk_length,

               top_p, temperature, repetition_penalty, seed):

    engine = get_engine()

    refs = []

    if ref_path and ref_text and ref_text.strip():

        with open(ref_path, "rb") as fh:

            audio_bytes = fh.read()

        refs = [ServeReferenceAudio(audio=audio_bytes, text=ref_text.strip())]

    req = ServeTTSRequest(

        text=text,

        references=refs,

        reference_id=None,

        max_new_tokens=int(max_new_tokens),

        chunk_length=int(chunk_length),

        top_p=float(top_p),

        temperature=float(temperature),

        repetition_penalty=float(repetition_penalty),

        seed=int(seed) if seed else None,

        use_memory_cache="on",

    )

    t0 = time.time()

    for result in engine.inference(req):

        if result.code == "final":

            sr, audio = result.audio

            return (int(sr), np.asarray(audio, dtype=np.float32)), None, f"Took {time.time() - t0:.1f}s"

        if result.code == "error":

            return None, str(result.error), None

    return None, "No audio generated.", None

print(f"[Fish] ready — device={DEVICE}, ckpt={CKPT_DIR}")

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app. Type or paste text, optionally upload a reference clip + its transcript
# @markdown for voice cloning, and click **Synthesize**. Inline `[tag]` markers control prosody / emotion.
with gr.Blocks(title="Fish Audio S2 Pro — Colab", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## Fish Audio S2 Pro — 4B+400M Dual-AR TTS\n"
        "80+ languages · 15 K inline tags · 4B slow AR + 400 M fast AR · ~21 Hz codec\n"
        "**Inline tag examples:** `[whisper]` · `[excited]` · `[low voice in small room]` · `[pitch up]` · `[laughing]` · `[pause]` · `[sad]`.\n"
        "**Multi-speaker:** prefix text with `<|speaker:0|>...<|speaker:1|>...` and provide matching ref clips.\n"
        "**Voice clone:** upload a 10–30 s reference clip and type what it says. The transcript is required."
    )
    with gr.Row():
        with gr.Column():
            text_in = gr.Textbox(
                label="Text to synthesize",
                lines=4,
                value="[excited] Hello! Welcome to Fish Audio S2 Pro, a 4 billion parameter multilingual TTS model with fifteen thousand inline emotion tags.",
            )
            with gr.Accordion("Voice cloning (optional)", open=False):
                ref_audio = gr.Audio(label="Reference audio (10–30 s clean speech)", type="filepath")
                ref_text = gr.Textbox(
                    label="Transcript of the reference audio",
                    lines=2,
                    placeholder="Type exactly what the reference clip says.",
                )
            with gr.Accordion("Advanced sampling", open=False):
                max_new_tokens = gr.Slider(0, 4096, value=0, step=64, label="max_new_tokens (0 = auto / until EOS)", info="Maximum audio length to generate. 0 = until the model says EOS.")
                chunk_length = gr.Slider(100, 1000, value=200, step=20, label="chunk_length (bytes per turn)", info="Bytes of text per generation turn. Higher = more context, lower = more responsive.")
                top_p = gr.Slider(0.1, 1.0, value=0.8, step=0.05, label="top_p", info="Nucleus sampling: only consider tokens whose cumulative probability is below this. 0.9 = off.")
                temperature = gr.Slider(0.1, 1.0, value=0.8, step=0.05, label="temperature", info="Lower = more deterministic, higher = more varied. 0.7 is a good default.")
                repetition_penalty = gr.Slider(0.9, 2.0, value=1.1, step=0.05, label="repetition_penalty", info="Penalize repeated tokens. 1.0 = no penalty, 1.1 = recommended.")
                seed = gr.Number(value=0, label="seed (0 = random)", precision=0)
            btn = gr.Button("Synthesize", variant="primary")
        with gr.Column():
            audio_out = gr.Audio(label="Generated audio", type="numpy")
            meta_out = gr.Markdown()
            err_out = gr.Markdown()
    btn.click(
        synthesize,
        inputs=[text_in, ref_audio, ref_text, max_new_tokens, chunk_length, top_p, temperature, repetition_penalty, seed],
        outputs=[audio_out, meta_out, err_out],
    )
    gr.Examples(
        examples=[
            ["[whisper in small voice] The quick brown fox jumps over the lazy dog.", None, ""],
            ["[excited] I just got the best news ever!", None, ""],
            ["[sad] Sometimes the hardest part isn't the work itself, but waiting for someone to notice it.", None, ""],
            ["[laughing] Oh no, you didn't!", None, ""],
        ],
        inputs=[text_in, ref_audio, ref_text],
    )
from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ Fish Audio S2 Pro ready. Add inline [tag] markers (e.g. [whisper], [excited]) to control prosody.", outputs=[meta_out])


In [ ]:
# @title Step 5 — Keep Alive

# @markdown Prevents Colab from disconnecting during long generation sessions. Run any time after Step 1.

import threading, time

def _keep():

    while True:

        time.sleep(60)

        try:

            import requests

            requests.get('https://www.google.com', timeout=5)

        except Exception:

            pass

threading.Thread(target=_keep, daemon=True).start()

print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single TTS call from the notebook. Useful for sanity-checking without Gradio.

TEXT = "[excited] The warm morning sun cast golden rays across the quiet valley, awakening the world with a gentle embrace."  # @param {type:"string"}
REF_AUDIO = ""  # @param {type:"string"}
REF_TEXT = ""  # @param {type:"string"}
TEMPERATURE = 0.8  # @param {type:"slider", min:0.1, max:1.5, step:0.05}
TOP_P = 0.8  # @param {type:"slider", min:0.1, max:1.0, step:0.05}
REPETITION_PENALTY = 1.1  # @param {type:"slider", min:0.9, max:2.0, step:0.05}
CHUNK_LENGTH = 200  # @param {type:"slider", min:100, max:1000, step:20}
MAX_NEW_TOKENS = 0  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}

if REF_AUDIO and (not REF_TEXT or not REF_TEXT.strip()):
    raise SystemExit("Reference audio is set but REF_TEXT is empty — the model needs the transcript for cloning.")
audio, err, dt = synthesize(
    TEXT, REF_AUDIO or None, REF_TEXT,
    MAX_NEW_TOKENS, CHUNK_LENGTH,
    TOP_P, TEMPERATURE, REPETITION_PENALTY, SEED,
)
if err:
    print(f"[Fish] error: {err}")
else:
    sr, wav = audio
    out_path = "/content/fish_s2pro_quick.wav"
    import soundfile as sf
    sf.write(out_path, wav, sr)
    print(f"[Fish] {len(wav)/sr:.2f}s @ {sr} Hz written to {out_path}  (took {dt:.1f}s)")
    from IPython.display import Audio, display
    display(Audio(out_path))


In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each tuple is `(text, ref_audio_path_or_None, ref_text_or_None)`.
# @markdown All outputs go to `/content/fish_s2pro_batch/` as 16-bit PCM WAVs.
BATCH = [
    ("[excited] Welcome to the Fish Audio S2 Pro model!", None, ""),
    ("[whisper] The library was perfectly silent.", None, ""),
    ("[angry] I told you three times already!", None, ""),
    ("Bonjour, je m'appelle Fish. Je suis un modèle de synthèse vocale.", None, ""),
]
BATCH_REF_AUDIO = None
BATCH_REF_TEXT = ""
BATCH_TEMPERATURE = 0.8
BATCH_TOP_P = 0.8
BATCH_REPETITION_PENALTY = 1.1
BATCH_CHUNK_LENGTH = 200
BATCH_MAX_NEW_TOKENS = 0
BATCH_SEED = 0
BATCH_OUT_DIR = "/content/fish_s2pro_batch"

import os, time, soundfile as sf
os.makedirs(BATCH_OUT_DIR, exist_ok=True)
for i, item in enumerate(BATCH):
    try:
        if len(item) == 3:
            text, ref_path, ref_text = item
        else:
            text = item[0]; ref_path = BATCH_REF_AUDIO; ref_text = BATCH_REF_TEXT
        if ref_path and (not ref_text or not ref_text.strip()):
            print(f"[Fish] {i+1}/{len(BATCH)} SKIP (ref audio set but ref text empty): {text[:60]}")
            continue
        print(f"[Fish] {i+1}/{len(BATCH)}: {text[:80]}")
        t0 = time.time()
        audio, err, dt = synthesize(
            text, ref_path, ref_text,
            BATCH_MAX_NEW_TOKENS, BATCH_CHUNK_LENGTH,
            BATCH_TOP_P, BATCH_TEMPERATURE, BATCH_REPETITION_PENALTY, BATCH_SEED,
        )
        if err or audio is None:
            print(f"  -> ERROR: {err}")
            continue
        sr, wav = audio
        safe = ''.join(c if c.isalnum() else '_' for c in text[:40]).strip('_') or f'item_{i:02d}'
        out_path = f"{BATCH_OUT_DIR}/{i:02d}_{safe}.wav"
        sf.write(out_path, wav, sr)
        print(f"  -> {len(wav)/sr:.2f}s @ {sr} Hz  ({dt:.1f}s)  -> {out_path}")
    except Exception as e:
        print(f"  -> EXCEPTION: {e}")
        continue
print(f"\n[Done] {len(BATCH)} items -> {BATCH_OUT_DIR}")